# Step 4 — MuJoCo Wipe-Table Evaluation

Dataset-oracle replay through the Step 2 CUDA ESN + proximity-gated cloth grasp + wipe metrics + video.

**Default:** all **40 held-out** episodes (`160–199`, `MAX_EPISODES=None`).  
Record **one** short MP4 (`table_wipe_ep160_oracle_esn.mp4`) via imageio/ffmpeg — not GIF.  
Cloth rest pose is taken from the demo's first grasp; table-contact uses a per-episode wipe plane.

This is **not** live UnifoLM. Tokens come from the demo dataset (oracle).

```bash
python3 -m src.step4_mujoco_evaluation --episodes heldout --no_video
python3 -m src.step4_mujoco_evaluation --episodes 160-162 --video_episode 160
```


In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
RESEARCH_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RESEARCH_DIR = RESEARCH_DIR.resolve()
os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step4_mujoco_evaluation'}")

Research root : /home/aihimekpen/research_summer_2026/research
Results go to : /home/aihimekpen/research_summer_2026/research/results/step4_mujoco_evaluation


In [2]:
EPISODES_SPEC = "heldout"     # heldout | train | 160-163 | 0
MAX_EPISODES = None            # None = all 40 held-out (160–199)
VIDEO_EPISODE = 160            # film one episode only
RECORD_VIDEO = True            # only VIDEO_EPISODE is filmed (MP4 via imageio/ffmpeg)
DURATION_S = None              # None = exactly one episode (~12 s)
LOOP_EPISODE = False           # never True for paper videos
CONTROL_MODE = "kinematic"     # kinematic | pd
CONTROL_HZ = 100.0
VLA_HZ = 2.0
VIDEO_FPS = 30.0
DEVICE = "cuda"
MJCF_PATH = None
ESN_CHECKPOINT = None

from src.wipe_dataset import parse_episode_spec, split_name_for_episodes
EPISODES = parse_episode_spec(EPISODES_SPEC)
if MAX_EPISODES is not None:
    EPISODES = EPISODES[:MAX_EPISODES]
SPLIT_TAG = split_name_for_episodes(EPISODES)
if VIDEO_EPISODE is None:
    VIDEO_EPISODE = EPISODES[0]
_dur = "1 episode" if DURATION_S is None else f"{DURATION_S:.0f}s"
print(f"split={SPLIT_TAG} | n={len(EPISODES)} | video_ep={VIDEO_EPISODE} | duration={_dur} | loop={LOOP_EPISODE}")


split=heldout | n=40 | video_ep=160 | duration=1 episode | loop=False


In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.paths import results_path
from src.step3_dual_thread_mujoco import resolve_esn_checkpoint, resolve_mjcf_path, load_esn_checkpoint_metadata
from src.step4_mujoco_evaluation import (
    MuJoCoEvalConfig,
    MuJoCoWipeEvaluator,
    print_eval_summary,
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required.")

mjcf = resolve_mjcf_path(MJCF_PATH)
ckpt = resolve_esn_checkpoint(ESN_CHECKPOINT)
meta = load_esn_checkpoint_metadata(ckpt)
out_dir = results_path("step4_mujoco_evaluation")

print("ESN train_episodes:", meta.get("train_episodes"))
print("ESN heldout_episodes:", meta.get("heldout_episodes"))
overlap = set(meta.get("train_episodes") or []) & set(EPISODES)
if overlap:
    print("WARNING: evaluating episodes that were in ESN train set:", sorted(overlap)[:10], "...")
else:
    print("OK: eval episodes are disjoint from ESN train_episodes (when metadata present).")

reports = []
for ep in EPISODES:
    record_video = bool(RECORD_VIDEO) and int(ep) == int(VIDEO_EPISODE)
    video_path = out_dir / f"table_wipe_ep{ep}_oracle_esn.mp4"
    config = MuJoCoEvalConfig(
        mjcf_path=mjcf,
        esn_checkpoint=str(ckpt),
        init_episode=int(ep),
        duration_s=DURATION_S,
        control_hz=CONTROL_HZ,
        vla_hz=VLA_HZ,
        control_mode=CONTROL_MODE,
        record_video=record_video,
        video_path=video_path if record_video else None,
        video_fps=VIDEO_FPS,
        device=DEVICE,
        loop_episode=LOOP_EPISODE,
    )
    stats = MuJoCoWipeEvaluator(config).run()
    report = {
        "init_episode": int(ep),
        "control_mode": CONTROL_MODE,
        "tracking_rmse": stats.tracking_rmse,
        "grasp_frames": stats.grasp_frames,
        "trajectory_steps": stats.trajectory_steps,
        "episode_table_top_z": stats.episode_table_top_z,
        "video_path": stats.video_path,
        "task_metrics": stats.task_metrics.to_dict() if stats.task_metrics else None,
        "esn_train_episodes": meta.get("train_episodes"),
    }
    ep_path = out_dir / f"mujoco_eval_report_ep{ep}.json"
    ep_path.write_text(json.dumps(report, indent=2))
    reports.append(report)
    print_eval_summary(stats, report_path=ep_path)

def _tm_mean(key, default=0.0):
    return float(np.mean([float((r.get("task_metrics") or {}).get(key, default)) for r in reports]))

def _tm_rate(key):
    return float(np.mean([1.0 if (r.get("task_metrics") or {}).get(key) else 0.0 for r in reports]))

summary = {
    "split": SPLIT_TAG,
    "episodes": EPISODES,
    "n_episodes": len(EPISODES),
    "tracking_rmse_mean": float(np.mean([r["tracking_rmse"] for r in reports])),
    "tracking_rmse_std": float(np.std([r["tracking_rmse"] for r in reports])),
    "grasp_success_rate": _tm_rate("grasp_success"),
    "task_success_rate": _tm_rate("task_success"),
    "wipe_path_m_mean": _tm_mean("wipe_path_length_m"),
    "table_contact_ratio_mean": _tm_mean("table_contact_ratio"),
    "wipe_coverage_m2_mean": _tm_mean("wipe_coverage_m2"),
    "reports": reports,
}
sum_path = out_dir / f"mujoco_eval_summary_{SPLIT_TAG}.json"
sum_path.write_text(json.dumps(summary, indent=2))
(out_dir / "mujoco_eval_report.json").write_text(json.dumps(summary, indent=2))

rows = []
for r in reports:
    tm = r.get("task_metrics") or {}
    rows.append({
        "episode": r["init_episode"],
        "tracking_rmse": r["tracking_rmse"],
        "grasp_success": tm.get("grasp_success"),
        "task_success": tm.get("task_success"),
        "wipe_path_m": tm.get("wipe_path_length_m"),
        "table_contact_ratio": tm.get("table_contact_ratio"),
        "wipe_coverage_m2": tm.get("wipe_coverage_m2"),
        "max_cloth_jump_m": tm.get("max_cloth_jump_m"),
        "video_path": r.get("video_path"),
    })
df = pd.DataFrame(rows)
csv_path = out_dir / f"mujoco_eval_summary_{SPLIT_TAG}.csv"
df.to_csv(csv_path, index=False)
display(df)
print(f"Summary: {sum_path}")
print(f"CSV:     {csv_path}")
print(
    f"RMSE {summary['tracking_rmse_mean']:.5f} ± {summary['tracking_rmse_std']:.5f} | "
    f"grasp={summary['grasp_success_rate']:.1%} | "
    f"contact={summary['table_contact_ratio_mean']:.1%} | "
    f"task_success={summary['task_success_rate']:.1%}"
)


In [5]:
from IPython.display import Video, display, Markdown
from pathlib import Path

vid = None
for r in reports:
    if r.get("video_path"):
        vid = r["video_path"]
        break
if vid and Path(vid).is_file():
    display(Markdown(f"**Benchmark video:** `{vid}`"))
    display(Video(vid, embed=True, width=640))
else:
    print("No video recorded this run (RECORD_VIDEO=False or export failed).")
    print("Tip: install imageio + imageio-ffmpeg in the UnifoLM venv for MP4.")


No video recorded this run (RECORD_VIDEO=False or export failed).
Tip: install imageio + imageio-ffmpeg in the UnifoLM venv for MP4.
